### Exercise 1

#### Part 1 & Part 2

Suppose the input is $X$. First apply kernel $K_1$, then apply kernel $K_2$:
$$
Y = X * K_1
$$

$$
Z = Y * K_2
$$

Substitute the first equation into the second:

$$
Z = (X * K_1) * K_2
$$

By associativity of convolution,

$$
Z = X * (K_1 * K_2)
$$

So the two-step operation is equivalent to one convolution with a new kernel

$$
K_{\text{new}} = K_1 * K_2
$$

If $K_1$ has size $k_1$ and $K_2$ has size $k_2$, then the effective kernel has size

$$
k_{\text{new}} = k_1 + k_2 - 1.
$$

For example, two $3 \times 3$ convolutions with no nonlinearity are equivalent to one $5 \times 5$ convolution, because

$$
3 + 3 - 1 = 5.
$$

Intuition: the first convolution lets each intermediate pixel depend on a local $k_1$-sized neighborhood. The second convolution then combines neighboring intermediate pixels over a $k_2$-sized window. So the final output depends on a larger region of the original input.

In 2D, if the kernels are $k_1 \times k_1$ and $k_2 \times k_2$, the effective kernel is

$$
(k_1 + k_2 - 1) \times (k_1 + k_2 - 1).
$$


#### Part 3

No. **Not every convolution kernel can be decomposed into two smaller convolution kernels.**

For 1D, suppose we want to decompose a length-3 kernel

$$
K = [a,b,c]
$$

into two length-2 kernels:

$$
U = [p,q], \qquad V = [r,s].
$$

Then

$$
U * V = [pr, ps+qr, qs].
$$

So we need

$$
a=pr,\qquad b=ps+qr,\qquad c=qs.
$$

This imposes structure. Some kernels satisfy it, but not all decompositions with smaller kernels are possible under extra constraints such as real-valued filters, fixed sizes, channel restrictions, or separability.

A clearer 2D example: a $3\times 3$ kernel cannot always be written as a $3\times 1$ kernel followed by a $1\times 3$ kernel. That would mean

$$
K = uv^\top,
$$

which is a rank-1 matrix. But most $3\times 3$ kernels have rank larger than 1.

So the answer is:

$$
\boxed{\text{Two smaller convolutions can always be combined into one larger convolution, but one larger convolution cannot always be decomposed into two smaller convolutions.}}
$$


### Exercise 2

#### Part 1

Output height and width are
$$
h_{\text {out }}=\left\lfloor\frac{h+2 p_h-k_h}{s_h}\right\rfloor+1, \quad w_{\text {out }}=\left\lfloor\frac{w+2 p_w-k_w}{s_w}\right\rfloor+1 .
$$

For each output element, one output channel looks at all $c_i$ input channels and a $k_h \times k_w$ spatial window. So each output element needs

$$
c_i k_h k_w
$$

multiplications, and roughly

$$
c_i k_h k_w - 1
$$

additions.

There are

$$
c_o h_{\text{out}} w_{\text{out}}
$$

output elements.

So the total cost is:

$$
\boxed{
\text{Multiplications} = c_o h_{\text{out}} w_{\text{out}} c_i k_h k_w
}
$$

and

$$
\boxed{
\text{Additions} = c_o h_{\text{out}} w_{\text{out}}(c_i k_h k_w - 1)
}
$$

Often we just say the forward cost is

$$
\boxed{
\mathcal{O}
\left(
c_o c_i h_{\text{out}} w_{\text{out}} k_h k_w
\right)
}
$$


#### Part 2

The memory footprint mainly has **three parts**.

Input tensor:

$$
c_i h w
$$

Kernel parameters:

$$
c_o c_i k_h k_w
$$

Output tensor:

$$
c_o h_{\text{out}} w_{\text{out}}
$$

where

$$
h_{\text {out }}=\left\lfloor\frac{h+2 p_h-k_h}{s_h}\right\rfloor+1, \quad w_{\text {out }}=\left\lfloor\frac{w+2 p_w-k_w}{s_w}\right\rfloor+1 .
$$

numbers.

If each number is a 32-bit float, multiply by 4 bytes:

$$
\boxed{
4\left(
c_i h w
+
c_o c_i k_h k_w
+
c_o h_{\text{out}} w_{\text{out}}
\right)
\text{ bytes}
}
$$

For training, memory is larger because we also need to store activations and gradients. Then we also keep things like:

$$
\nabla X,\quad \nabla K,\quad \nabla Y
$$

so the memory is usually several times the forward-only footprint.

#### Part 3

For backward computation, we usually need to keep:

$$
X, \quad K, \quad Y
$$

from the forward pass, and compute/store gradients:

$$
\frac{\partial L}{\partial Y}, \quad \frac{\partial L}{\partial X}, \quad \frac{\partial L}{\partial K} .
$$


So the main memory footprint is approximately:

$$
X+K+Y+\nabla Y+\nabla X+\nabla K
$$


In terms of tensor sizes:

$$
\begin{gathered}
X: c_i h w \\
K: c_o c_i k_h k_w \\
Y: c_o h_{\text {out }} w_{\text {out }} \\
\nabla Y: c_o h_{\text {out }} w_{\text {out }} \\
\nabla X: c_i h w \\
\nabla K: c_o c_i k_h k_w
\end{gathered}
$$
Therefore,

$$
2 c_i h w+2 c_o c_i k_h k_w+2 c_o h_{\text {out }} w_{\text {out }}
$$

numbers.
If using 32-bit floats, multiply by 4 bytes:

$$
4\left(2 c_i h w+2 c_o c_i k_h k_w+2 c_o h_{\text {out }} w_{\text {out }}\right) \text { bytes }
$$

#### Part 4

For backpropagation through a convolution, we usually compute two gradients:

$$
\frac{\partial L}{\partial K} \quad \text { and } \quad \frac{\partial L}{\partial X} .
$$


Recall the forward cost was roughly

$$
c_o c_i h_{\text {out }} w_{\text {out }} k_h k_w
$$

multiplications, plus a similar number of additions.
For the kernel gradient $\partial L / \partial K$, each kernel weight looks at all spatial output positions, so the cost is approximately

$$
c_o c_i k_h k_w h_{\text {out }} w_{\text {out }}
$$

multiplications and roughly the same number of additions.
For the input gradient $\partial L / \partial \mathrm{X}$, we propagate gradients from output locations back through the same kernel weights. This has approximately the same cost:

$$
c_o c_i k_h k_w h_{\text {out }} w_{\text {out }}
$$

multiplications and roughly the same number of additions.
So total backpropagation cost is approximately

$$
2 c_o c_i k_h k_w h_{\text {out }} w_{\text {out }}
$$
multiplications, plus a similar number of additions.
Including the forward pass, training one layer costs roughly

$$
3 c_o c_i k_h k_w h_{\text {out }} w_{\text {out }}
$$

multiply-add work.
Here,

$$
h_{\text {out }}=\left\lfloor\frac{h+2 p_h-k_h}{s_h}\right\rfloor+1, \quad w_{\text {out }}=\left\lfloor\frac{w+2 p_w-k_w}{s_w}\right\rfloor+1 .
$$


Intuition: backward is about twice as expensive as forward, because we need one convolution-like operation for the weights and one convolution-like operation for the input.

### Exercise 3

The forward computational cost is

$$
\mathcal{O}\left(c_o c_i h_{\text {out }} w_{\text {out }} k_h k_w\right) .
$$


So if we double both channels,
Ask ChatGPT

$$
c_i \rightarrow 2 c_i, \quad c_o \rightarrow 2 c_o,
$$

then the cost becomes

$$
\left(2 c_o\right)\left(2 c_i\right) h_{\text {out }} w_{\text {out }} k_h k_w=4 c_o c_i h_{\text {out }} w_{\text {out }} k_h k_w .
$$


So the number of calculations increases by

$$
4 x
$$

assuming output spatial size stays the same.
For padding, recall

$$
h_{\text {out }}=\left\lfloor\frac{h+2 p_h-k_h}{s_h}\right\rfloor+1, \quad w_{\text {out }}=\left\lfloor\frac{w+2 p_w-k_w}{s_w}\right\rfloor+1 .
$$
If we double padding,

$$
p_h \rightarrow 2 p_h, \quad p_w \rightarrow 2 p_w,
$$

then

$$
h_{\text {out }}=\left\lfloor\frac{h+4 p_h-k_h}{s_h}\right\rfloor+1, \quad w_{\text {out }}=\left\lfloor\frac{w+4 p_w-k_w}{s_w}\right\rfloor+1 .
$$


So padding affects computation indirectly by changing the output height and width. The cost increases by the factor

$$
\frac{h_{\text {out }}^{\prime} w_{\text {out }}^{\prime}}{h_{\text {out }} w_{\text {out }}}
$$

where $h_{\text {out }}^{\prime}, w_{\text {out }}^{\prime}$ are the output sizes after doubling padding.
So:
Doubling $c_i$ and $c_o \Rightarrow 4 \times$ cost
Doubling padding ⇒ cost increases only if output spatial size increases.

### Exercise 5

1. Key idea

A convolution does this:
- For each output location → take a local patch of the input
- Flatten it → take a dot product with the kernel

That is already a matrix multiplication pattern:
- each patch = one row
- each kernel = one column (or row depending on convention)

2. Single-channel intuition (simplest case)

Assume:
- Input: $h \times w$
- Kernel: $k_h \times k_w$
- Output: $h_{\text {out }} \times w_{\text {out }}$

Step 1: Extract patches (im2col)

Each sliding window gives a vector of size:

$$
k_h k_w
$$


There are:

$$
h_{\text {out }} w_{\text {out }}
$$
such patches.
So we build a matrix:

$$
X_{\mathrm{col}} \in \mathbb{R}^{\left(h_{\mathrm{out}} w_{\mathrm{out}}\right) \times\left(k_h k_w\right)}
$$


Each row = one flattened patch.

Step 2: Flatten kernel

$$
K_{\text {col }} \in \mathbb{R}^{\left(k_h k_w\right) \times 1}
$$


Step 3: Matrix multiplication

$$
\begin{gathered}
Y_{\mathrm{col}}=X_{\mathrm{col}} K_{\mathrm{col}} \\
Y_{\mathrm{ccl}} \in \mathbb{R}^{\left(h_{\mathrm{out}} w_{\mathrm{out}}\right) \times 1}
\end{gathered}
$$


Reshape back:

$$
Y \in \mathbb{R}^{h_{\text {out }} \times w_{\text {out }}}
$$

3. Multi-channel + multi-output (general case)

Now the real setting:
- Input: $c_i \times h \times w$
- Kernel: $c_o \times c_i \times k_h \times k_w$

Step 1: im2col

Each patch now includes all channels:

$$
\text { patch size }=c_i k_h k_w
$$


Matrix:

$$
X_{\text {col }} \in \mathbb{R}^{\left(h_{\text {out }} w_{\text {out }}\right) \times\left(c_i k_h k_w\right)}
$$


Step 2: reshape kernels

Each output channel has one kernel → flatten it:

$$
K_{\text {col }} \in \mathbb{R}^{\left(c_i k_h k_w\right) \times c_o}
$$

Step 3: matrix multiplication

$$
\begin{gathered}
Y_{\mathrm{col}}=X_{\mathrm{col}} K_{\mathrm{col}} \\
Y_{\mathrm{col}} \in \mathbb{R}^{\left(h_{\mathrm{out}} w_{\mathrm{out}}\right) \times c_o}
\end{gathered}
$$


Step 4: reshape output

$$
Y \in \mathbb{R}^{c_o \times h_{\text {out }} \times w_{\text {out }}}
$$

4. Final boxed result

$$
\text { Convolution } \equiv \operatorname{im} 2 \operatorname{col}(X) \times \operatorname{reshape}(K)
$$

where

$$
\left(h_{\text {out }} w_{\text {out }} \times c_i k_h k_w\right) \times\left(c_i k_h k_w \times c_o\right)
$$

### Exercise 6

#### Part 1

Suppose the original matrix is $c \times c$, and we multiply it by a vector of length $c$.
A dense matrix-vector multiplication costs roughly

$$
c^2
$$

multiplications.
Now break the matrix into $b$ equal block-diagonal blocks. Each block has size

$$
\frac{c}{b} \times \frac{c}{b} .
$$


Each block multiplication costs

$$
\left(\frac{c}{b}\right)^2 .
$$


There are $b$ blocks, so total cost is

$$
b\left(\frac{c}{b}\right)^2=\frac{c^2}{b} .
$$


So compared with dense multiplication:

$$
\frac{c^2}{c^2 / b}=b .
$$


Therefore, it is

$b$ times faster

assuming equal block sizes and ignoring overhead.

#### Part 2

1. What happens when you use block-diagonal structure?

From the previous result, you gain:

$$
\operatorname{cost} \sim \frac{c^2}{b}
$$

much cheaper computation.
But what did we remove?
A block-diagonal matrix looks like:

$$
\left[\begin{array}{ccc}
\boldsymbol{A}_1 & 0 & 0 \\
0 & \boldsymbol{A}_2 & 0 \\
0 & 0 & \boldsymbol{A}_3
\end{array}\right]
$$


So each block only interacts with its own part of the input.

2. The downside

No interaction across blocks
That means:
- Information in block 1 never affects block 2
- The model cannot learn cross-channel / cross-feature relationships
- This severely limits expressiveness.

In neural network terms:
- Each block is like an independent sub-network
- You lose the ability to combine features globally

3. Why this is bad (intuition)

Suppose:
- Block 1 detects edges
- Block 2 detects textures

Without interaction:
- You cannot combine them into "edge-with-texture" features

Model becomes fragmented.


4. How to fix it (partially)
✓ Idea 1: Add mixing layers (most important)

Insert a dense or $1 \times 1$ convolution layer:
- This mixes information across blocks
- Restores cross-channel interaction

This is exactly what modern architectures do:
- ResNeXt
- MobileNet
- ShuffleNet
✓ Idea 2: Shuffle channels

Instead of fixed blocks:
- permute channels between layers

So next layer sees mixed inputs.
Used in ShuffleNet